### Business Understanding and Planning Phase
Current Manual Process and Limitations- Loan aprroavals are currently reviewed manually buy loan officers.
The limitations of these include:- Inconsistent decisions due to subjective  judgement, slow response times which can lead to poor customer experience, Potential bias where some credit worthy applicants may be overlooked and scalability issues where manual review doesnt scale with growth.

Some of the key stakeholders and their needs include-
Loan officers - they need standardized, data-driven support to reduce bias and speed up decisions.
Applicants - they want a fast, fair and transparent loan approvals.
Risk Analytics Team - need robust, interpretable models to balance profit vs risk.
Regulators - require transparency anf explainability in credit decisions.
Executives - want improved profitability and reduced default losses.

Some o fthe implications of model Errors are: False approval(approve a bad loan) - very costly (~$50 000 loss)
False denial(reject a good loan) - moderate cost(~$8000 lost profit)
Precision on approvals is more critical than recall but both matter.

So, our choice of approach is regression where we predict the risk score over classification since this produces a continous probability of default.
Loan officers can also set thr threshholds dynamilcally based on the risk appetite of the business.
It is also more flexible for business use and reulatory reporting.
It also supports human-in-the-loop decision making.

### Modeling Goals and Sucess Criteria
The goal of the model will be to predict the applicant ris score(likelihood of default).
The sucess creteria will include: Minimizing the expected financial loss which will be done using a custom cost metric.
Achiving a strong discrimination ability(ROC-AUC >= 0.75)
Ensuring well calibrated probabilities(Brier score <= 0.20)
Providing interpretable outputs(SHAP values, feature importance)

### Evaluation Metrics
ROC-AUC - measures ability to separate good vs bad loans
Brier Score - measures calibration of predicted probabilities
Custom Expected Cost Metric: Expected Loss= (FP.50000) + (FN.8000)
It ties directly into the model evaluation business impact.

### Baseline Performance targets
ROC-AUC >= 0.75 which is better than random guessing at 0.5.
Brier Score <= 0.20 which implies that the model is well calibrated.
Expected Loss should be lower that current manual process(benchmark to be established from historical officers decisions).

In [1]:
# importing the necessary libraries
import pandas as pd

In [2]:
df = pd.read_csv("financial_loan_data.csv")
df.head()

,Age,AnnualIncome,CreditScore,EmploymentStatus,EducationLevel,Experience,LoanAmount,LoanDuration,MaritalStatus,NumberOfDependents,HomeOwnershipStatus,MonthlyDebtPayments,CreditCardUtilizationRate,NumberOfOpenCreditLines,NumberOfCreditInquiries,DebtToIncomeRatio,BankruptcyHistory,LoanPurpose,PreviousLoanDefaults,PaymentHistory,LengthOfCreditHistory,SavingsAccountBalance,CheckingAccountBalance,TotalAssets,TotalLiabilities,MonthlyIncome,UtilityBillsPaymentHistory,JobTenure,NetWorth,BaseInterestRate,InterestRate,MonthlyLoanPayment,TotalDebtToIncomeRatio,LoanApproved,RiskScore
0,45,"$39,948.00",617,Employed,Master,22,13152,48,Married,2,Own,183,0.354418,1,2,0.358336,No,Home,0,29,9,7632.0,1202,146111,19183,3329.000000,0.724972,11,126928,0.199652,0.227590,419.805992,0.181077,0,49.0
1,38,"$39,709.00",628,Employed,Associate,15,26045,48,Single,1,Mortgage,496,0.087827,5,3,0.330274,No,Debt Consolidation,0,21,9,4627.0,3460,53204,9595,3309.083333,0.935132,3,43609,0.207045,0.201077,794.054238,0.389852,0,52.0
2,47,"$40,724.00",570,Employed,Bachelor,26,17627,36,NaN,2,Rent,902,0.137414,2,0,0.244729,No,Education,0,20,22,886.0,895,25176,128874,3393.666667,0.872241,6,5205,0.217627,0.212548,666.406688,0.462157,0,52.0
3,58,"$69,084.00",545,Employed,High School,34,37898,96,Single,1,Mortgage,755,0.267587,2,1,0.436244,No,Home,0,27,10,1675.0,1217,104822,5370,5757.000000,0.896155,5,99452,0.300398,0.300911,1047.506980,0.313098,0,54.0
4,37,"$103,264.00",594,Employed,Associate,17,9184,36,Married,1,Mortgage,274,0.320535,0,0,0.078884,No,Debt Consolidation,0,26,27,1555.0,4981,244305,17286,8605.333333,0.941369,5,227019,0.197184,0.175990,330.179140,0.070210,1,36.0


First, we will be carrying out EDA and data cleacing to ensure we handle missing values.

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 35 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Age                         20000 non-null  int64  
 1   AnnualIncome                20000 non-null  object 
 2   CreditScore                 20000 non-null  int64  
 3   EmploymentStatus            20000 non-null  object 
 4   EducationLevel              19099 non-null  object 
 5   Experience                  20000 non-null  int64  
 6   LoanAmount                  20000 non-null  int64  
 7   LoanDuration                20000 non-null  int64  
 8   MaritalStatus               18669 non-null  object 
 9   NumberOfDependents          20000 non-null  int64  
 10  HomeOwnershipStatus         20000 non-null  object 
 11  MonthlyDebtPayments         20000 non-null  int64  
 12  CreditCardUtilizationRate   20000 non-null  float64
 13  NumberOfOpenCreditLines     200

In [4]:
print(df.shape)
print(df.dtypes)
df.isnull().sum().sort_values(ascending=False)

(20000, 35)
Age                             int64
AnnualIncome                   object
CreditScore                     int64
EmploymentStatus               object
EducationLevel                 object
Experience                      int64
LoanAmount                      int64
LoanDuration                    int64
MaritalStatus                  object
NumberOfDependents              int64
HomeOwnershipStatus            object
MonthlyDebtPayments             int64
CreditCardUtilizationRate     float64
NumberOfOpenCreditLines         int64
NumberOfCreditInquiries         int64
DebtToIncomeRatio             float64
BankruptcyHistory              object
LoanPurpose                    object
PreviousLoanDefaults            int64
PaymentHistory                  int64
LengthOfCreditHistory           int64
SavingsAccountBalance         float64
CheckingAccountBalance          int64
TotalAssets                     int64
TotalLiabilities                int64
MonthlyIncome                 float64


MaritalStatus                 1331
EducationLevel                 901
SavingsAccountBalance          572
CreditScore                      0
AnnualIncome                     0
Age                              0
Experience                       0
LoanAmount                       0
LoanDuration                     0
NumberOfDependents               0
EmploymentStatus                 0
MonthlyDebtPayments              0
CreditCardUtilizationRate        0
NumberOfOpenCreditLines          0
NumberOfCreditInquiries          0
DebtToIncomeRatio                0
BankruptcyHistory                0
LoanPurpose                      0
HomeOwnershipStatus              0
PreviousLoanDefaults             0
PaymentHistory                   0
LengthOfCreditHistory            0
CheckingAccountBalance           0
TotalAssets                      0
TotalLiabilities                 0
MonthlyIncome                    0
UtilityBillsPaymentHistory       0
JobTenure                        0
NetWorth            

In [5]:
import matplotlib.pyplot as plt
import seaborn as sns

### Data Understanding and exploration
Basic data characteristics include:
20000 loan applications
35 feature: mix of numerical(18) and categorical(7)
Targets are riskscore(continous), loanapproaved(binary)
missing values:
Marital Status(6.7%)
Education Level(4.5%)
SavingsAccountBalance(2.9%)
NO missingness in other features

For the marital status, we will replace the missing values with unknown as it might be indicative of the loan worthiness of an applicant and we would like to preserve it as a signal. Education also often correlates with repayment  ability and therefore we will also impute the missing values with the unnknown.
For the Savings account balance, a missing value might imply  a no-savings acoount and therefore  we will instead replace the missing values with zero. This 